Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC WITH daily_frame AS (
# MAGIC   SELECT day_date
# MAGIC   FROM (
# MAGIC     SELECT sequence(
# MAGIC       date '2025-01-01',
# MAGIC       current_date - interval '1' day,
# MAGIC       interval '1' day
# MAGIC     ) AS days
# MAGIC   )
# MAGIC   CROSS JOIN UNNEST(days) AS t(day_date)
# MAGIC )
# MAGIC
# MAGIC SELECT 
# MAGIC     DATE_FORMAT(day_date,'%Y-%m') AS year_month,
# MAGIC     SUM(sale) AS sale_count,
# MAGIC     COUNT(DISTINCT CASE 
# MAGIC         WHEN CAST(expiry_date AS DATE) >= CAST(day_date AS DATE) 
# MAGIC         THEN navi_customer_id 
# MAGIC     END) AS customer_count
# MAGIC FROM
# MAGIC (
# MAGIC     SELECT
# MAGIC         day_date,
# MAGIC         expiry_date,
# MAGIC         navi_customer_id,
# MAGIC
# MAGIC         COUNT(DISTINCT CASE 
# MAGIC             WHEN sale_ist2 IS NOT NULL 
# MAGIC             AND DATE(day_date) = DATE(sale_ist2) 
# MAGIC             THEN sale_policy_id2 
# MAGIC         END) AS sale
# MAGIC
# MAGIC     FROM
# MAGIC     (
# MAGIC         SELECT 
# MAGIC             j.*,
# MAGIC             day_date,
# MAGIC             date_diff('day',date(cast(call_connect_ist as timestamp)), day_date) AS dpd
# MAGIC         FROM daily_frame
# MAGIC
# MAGIC         LEFT JOIN
# MAGIC         (
# MAGIC             SELECT 
# MAGIC                 bt.*,
# MAGIC                 sale.sale_ist AS sale_ist2,
# MAGIC                 sale.sale_policy_id AS sale_policy_id2,
# MAGIC
# MAGIC                 CASE 
# MAGIC                     WHEN sale.sale_ist IS NOT NULL 
# MAGIC                     THEN date(cast(sale.sale_ist AS timestamp))
# MAGIC                     ELSE date(cast(bt.call_connect_ist AS timestamp)) + interval '60' day
# MAGIC                 END AS expiry_date
# MAGIC
# MAGIC             FROM corp_analytics.gi_jarvis_datamodel_cte_tbl bt
# MAGIC
# MAGIC             LEFT JOIN corp_analytics.gi_jarvis_sale_master_base_tbl sale
# MAGIC             ON sale.crm_ticket_id = bt.crm_ticket_id
# MAGIC
# MAGIC             WHERE date(cast(bt.call_connect_ist AS timestamp)) 
# MAGIC                   BETWEEN current_date - interval '9' month AND current_date
# MAGIC             AND lower(bt.relevancy)='relevant'
# MAGIC         ) j
# MAGIC
# MAGIC         ON date(day_date) <= date(expiry_date)
# MAGIC         AND date(day_date) BETWEEN date(cast(j.call_connect_ist AS timestamp))
# MAGIC         AND date(cast(j.call_connect_ist AS timestamp)) + interval '60' day
# MAGIC     ) base
# MAGIC
# MAGIC     GROUP BY 
# MAGIC         day_date,
# MAGIC         expiry_date,
# MAGIC         navi_customer_id
# MAGIC ) t
# MAGIC
# MAGIC WHERE day_date BETWEEN current_date - interval '90' day 
# MAGIC AND current_date - interval '1' day
# MAGIC
# MAGIC GROUP BY DATE_FORMAT(day_date,'%Y-%m')
# MAGIC ORDER BY year_month DESC
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)
# MAGIC else:
# MAGIC     print(f"No annotation data found for today ({pd.Timestamp.now().strftime('%Y-%m-%d')}). The DataFrame has {len(df.columns)} columns and 0 rows.")
# MAGIC     display(spark.createDataFrame([], schema="dummy STRING").limit(0))

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1FoHpHPMlw5HclJygb48_cLbHT6Lu5rV7be23DweUerI/edit?pli=1&gid=2067995280#gid=2067995280"
worksheet_name = "Focus Metric Data"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=2
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=2
)